In [1]:
import pandas as pd
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Custom modules
sys.path.append(('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/datasets/spe-1/spe1_helper_modules/'))
from spk_feat_cluster_comp_analysis  import *

In [2]:
cluster_pickle_dir = "/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spe1_pickles/cluster_pickles/"

### Step 1: Create dataframe with all experiment metadata and clustering results

In [3]:
#create df for all experiments 
df_master = compile_experiment_results(cluster_pickle_dir)


In [4]:
df_master

,cell_id,patch_type,current_type,cell_type,cortical_depth,dark_neuron,clear_EAP_waveform,spike_feature,num_clusters,cluster,nRMSE,cos_sim
0,c1,Juxta,IC,PC,843.9,False,False,peak_amp,3,Low-High,0.097350,0.990636
1,c1,Juxta,IC,PC,843.9,False,False,peak_sharpness,3,Low-High,0.089557,0.994296
2,c1,Juxta,IC,PC,843.9,False,False,exp_lambda,3,Low-High,0.057307,0.991058
3,c1,Juxta,IC,PC,843.9,False,False,log_isi,3,Low-Mid,0.042693,0.998962
4,c1,Juxta,IC,PC,843.9,False,False,log_isi,3,Low-High,0.039721,0.999174
...,...,...,...,...,...,...,...,...,...,...,...,...
122,c45,Juxta,VC,PC,1137.0,False,True,exp_lambda,3,Low-Mid,0.098120,0.906886
123,c45,Juxta,VC,PC,1137.0,False,True,exp_lambda,3,Low-High,0.057813,0.970601
124,c45,Juxta,VC,PC,1137.0,False,True,exp_lambda,3,Mid-High,0.044018,0.980513
125,c46,Juxta,VC,PC,1039.7,False,True,exp_lambda,2,Low-High,0.008720,0.999815


In [5]:
#sort by cell id
# Create a numeric column for sorting (c1, c2, c10, etc.)
df_master['sort_idx'] = df_master['cell_id'].str.extract('(\d+)').astype(int)

# Sort by the numeric ID, then by feature name
df_master = df_master.sort_values(by=['sort_idx', 'spike_feature']).drop(columns=['sort_idx'])



### Step 2: Generate table figure from dataframe 


In [ ]:

gen_table_fig(df_master)


### Step 3: Patterns for clustering/metadata

#### A) Do metadata features correlate with clustering waveform quantification (nRMSE, cos sim, number of clusters)

In [ ]:
sig_meta_clust_feats = analyze_cross_correlations(df_master)
sig_meta_clust_feats

#### Explore the pairs that are correlated

In [ ]:
def drill_down_significant_pairs(df, sig_pairs_df):
    """
    Generates detailed plots for each significant Metadata-Feature pair.
    Handles both categorical and continuous metadata automatically.
    """
    for _, row in sig_pairs_df.iterrows():
        meta_col = row['Metadata']
        feat_col = row['Feature']
        r_val = row['Pearson $r$']
        p_val = row['p-value']
        
        plt.figure(figsize=(9, 6))
        
        # Check if the metadata is continuous (e.g., depth) or categorical
        # Usually, if it has more than 10 unique values, we treat it as continuous
        is_continuous = df[meta_col].nunique() > 10
        
        if is_continuous:
            # Scatter plot with regression line for continuous data
            sns.regplot(data=df, x=meta_col, y=feat_col, 
                        scatter_kws={'alpha':0.4, 'color':'black', 's':20},
                        line_kws={'color':'#7b3294', 'lw':3})
        else:
            # Boxplot for categorical data
            ax = sns.boxplot(data=df, x=meta_col, y=feat_col, 
                             palette='PRGn', showfliers=False)
            
            # Manually set alpha for the boxes to avoid the TypeError
            for patch in ax.artists:
                r, g, b, a = patch.get_facecolor()
                patch.set_facecolor((r, g, b, 0.4))
                
            sns.stripplot(data=df, x=meta_col, y=feat_col, 
                          color='black', alpha=0.4, size=4)
        
        plt.title(f"{meta_col} vs {feat_col}", fontsize=14, fontweight='bold')
        plt.suptitle(f"Pearson $r = {r_val}$ | $p = {p_val}$", fontsize=10, y=0.92)
        plt.grid(axis='y', linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()



# Run it on the results from the previous step
drill_down_significant_pairs(df_master, sig_meta_clust_feats)

#### B) Study patterns in the spike features that cluster

#### C) What are the experiments with the biggest difference in waveform (nRMSE) and waveform abs shape (cos sim)

In [ ]:
top_var_df = analyze_waveform_variance(df_master, N=30)

In [ ]:
top_var_df 